# Иерархический CatBoost для прогнозирования продаж Favorita

В этом ноутбуке мы реализуем ML-подход с использованием CatBoost на агрегированных рядах.

In [ ]:
import os
import pickle
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append('..')
from src.models.catboost_adapter import CatBoostHierarchicalAdapter
from src.models.hierarchical_pipeline import HierarchicalForecaster
from src.utils.metrics import nwrmsle

In [ ]:
warnings.filterwarnings('ignore')

In [2]:
RAW_DATA_DIR = r"..\data\raw_data"
PREP_DATA_DIR = r"..\data\prepared_data"

In [3]:
train = pd.read_parquet(os.path.join(PREP_DATA_DIR, "train_filled_last3m.parquet"))
items = pd.read_csv(os.path.join(RAW_DATA_DIR, "items.csv"))
stores = pd.read_csv(os.path.join(RAW_DATA_DIR, "stores.csv"))

In [4]:
# Внешние данные
oil = pd.read_csv(os.path.join(RAW_DATA_DIR, "oil.csv"), parse_dates=['date'])
transactions = pd.read_csv(os.path.join(RAW_DATA_DIR, "transactions.csv"), parse_dates=['date'])

In [5]:
train = train[train['date'] >= '2017-05-20']
oil = oil[oil['date'] >= '2017-05-20']
transactions = transactions[transactions['date'] >= '2017-05-20']

In [6]:
print(f"train: {train.shape}, период: {train['date'].min()} -> {train['date'].max()}")
print(f"items: {items.shape}, stores: {stores.shape}")

train: (13801332, 5), период: 2017-05-20 00:00:00 -> 2017-08-14 00:00:00
items: (4100, 4), stores: (54, 5)


In [7]:
# Подготавливаем веса для метрики
item_weights = items.set_index('item_nbr')['perishable'].to_dict()
item_weights = {k: 1.25 if v == 1 else 1.0 for k, v in item_weights.items()}

# %%
# Разделяем на train/val
train_end_date = pd.Timestamp('2017-07-29')
val_end_date = pd.Timestamp('2017-08-14')

train_data = train[train['date'] <= train_end_date].copy()
val_data = train[(train['date'] > train_end_date) & (train['date'] <= val_end_date)].copy()

# Добавляем веса в валидацию
val_data['weight'] = val_data['item_nbr'].map(item_weights).fillna(1.0)

print(f"Train: {len(train_data)} записей, с {train_data['date'].min()} по {train_data['date'].max()}")
print(f"Val: {len(val_data)} записей, с {val_data['date'].min()} по {val_data['date'].max()}")
print(f"Уникальных пар в train: {train_data.groupby(['store_nbr', 'item_nbr']).ngroups}")
print(f"Уникальных пар в val: {val_data.groupby(['store_nbr', 'item_nbr']).ngroups}")

Train: 11263156 записей, с 2017-05-20 00:00:00 по 2017-07-29 00:00:00
Val: 2538176 записей, с 2017-07-30 00:00:00 по 2017-08-14 00:00:00
Уникальных пар в train: 158636
Уникальных пар в val: 158636


In [8]:
def prepare_external_features(train_data, val_data, stores, items):
    """
    Подготавливает внешние признаки для использования в модели.
    """
    external = {}
    
    # 1. Нефть (дневные цены)
    oil_features = oil.copy()
    oil_features.columns = ['date', 'oil_price']
    oil_features['oil_price_lag1'] = oil_features['oil_price'].shift(1)
    oil_features['oil_price_lag7'] = oil_features['oil_price'].shift(7)
    oil_features['oil_price_change'] = oil_features['oil_price'].pct_change()
    external['oil'] = oil_features
    
    # 3. Транзакции по магазинам
    # Добавляем информацию о кластере
    trans_with_cluster = transactions.merge(
        stores[['store_nbr', 'cluster']], 
        on='store_nbr', 
        how='left'
    )
    # Агрегируем по кластерам
    cluster_trans = trans_with_cluster.groupby(['date', 'cluster'])['transactions'].sum().reset_index()
    external['transactions'] = cluster_trans
    
    return external

In [9]:
external_data = prepare_external_features(train_data, val_data, stores, items)
print("Внешние признаки подготовлены:")
for key in external_data:
    print(f"  {key}: {external_data[key].shape}")

Внешние признаки подготовлены:
  oil: (74, 5)
  transactions: (1496, 3)


In [10]:
def create_catboost_adapter(horizon: int = 16, history: int = 90, use_external: bool = True):
    """Фабрика для создания адаптеров CatBoost"""
    return CatBoostHierarchicalAdapter(
        model_params={
            'iterations': 1000,
            'learning_rate': 0.05,
            'depth': 6,
            'loss_function': 'RMSE',
            'random_seed': 42,
            'verbose': False,
            'early_stopping_rounds': 50
        },
        horizon=horizon,
        history=history,
        use_external_features=use_external
    )

In [ ]:
# Параметры
horizon = (val_data['date'].max() - val_data['date'].min()).days + 1
print(f"Горизонт прогноза: {horizon} дней")


Горизонт прогноза: 16 дней


In [ ]:
def create_catboost_for_level(level_name=None):
    """Создает адаптер CatBoost с параметрами, специфичными для уровня иерархии"""
    adapter = CatBoostHierarchicalAdapter(
        model_params={
            'iterations': 100,
            'learning_rate': 0.01,
            'depth': 5,
            'loss_function': 'RMSE',
            'random_seed': 42,
            'verbose': False,
            'early_stopping_rounds': 10,
            'task_type': "GPU"
        },
        horizon=horizon,
        history=60,
        use_external_features=True
    )
    if level_name:
        adapter.set_level_name(level_name)
    return adapter

In [ ]:
# Создаем иерархический прогнозист
hier_forecaster = HierarchicalForecaster(stores, items)

Создание иерархии...
  Всего уникальных товаров: 3934
  Всего уникальных магазинов: 54
  Средние продажи total: 5.43
Пропорции вычислены:
  Средние продажи (total): 5.43
  Уникальных store_proportions: 54
  Уникальных item_proportions: 54279
  Total: 71 дней
  By cluster: 17 кластеров
  By cluster+family: 546 групп


In [ ]:
# Создаем иерархию
aggregated_series = hier_forecaster.create_hierarchy(train_data)

In [14]:
start_time = time.time()

# 1. Обучаем на total уровне
print("\n--- Уровень Total ---")
total_series = aggregated_series['total'].rename(columns={'total_sales': 'value'})
total_val = total_series[total_series['date'] >= train_end_date - pd.Timedelta(days=60)]

total_model = create_catboost_for_level('total')
total_model.fit(
    train_data=total_series[total_series['date'] <= train_end_date],
    val_data=total_val,
    external_data=external_data
)
hier_forecaster.trained_models['total'] = total_model

# 2. Обучаем на cluster уровне
print("\n--- Уровень Clusters ---")
cluster_models = {}
clusters_list = list(aggregated_series['by_cluster']['cluster'].unique())
for idx, cluster in enumerate(clusters_list):
    print(f"  Кластер {cluster} ({idx+1}/{len(clusters_list)})")
    cluster_data = aggregated_series['by_cluster'][
        aggregated_series['by_cluster']['cluster'] == cluster
    ].rename(columns={'unit_sales': 'value'}).sort_values('date')
    
    if len(cluster_data) < 30:
        print(f"    Пропускаем (мало данных: {len(cluster_data)} дней)")
        continue
        
    cluster_val = cluster_data[cluster_data['date'] >= train_end_date - pd.Timedelta(days=60)]
    
    cluster_model = create_catboost_for_level(f'cluster_{cluster}')
    cluster_model.fit(
        train_data=cluster_data[cluster_data['date'] <= train_end_date],
        val_data=cluster_val,
        external_data=external_data
    )
    cluster_models[cluster] = cluster_model
    
hier_forecaster.trained_models['clusters'] = cluster_models

# 3. Обучаем на cluster+family уровне
print("\n--- Уровень Cluster+Family ---")
cf_models = {}
cf_groups = list(aggregated_series['by_cluster_family'].groupby(['cluster', 'family']))
total_groups = len(cf_groups)

for idx, ((cluster, family), group) in enumerate(cf_groups):
    if idx % 50 == 0:
        print(f"  Прогресс: {idx}/{total_groups}")
    
    cf_data = group.rename(columns={'unit_sales': 'value'}).sort_values('date')
    
    if len(cf_data) < 30:
        continue
        
    cf_val = cf_data[cf_data['date'] >= train_end_date - pd.Timedelta(days=60)]
    
    cf_model = create_catboost_for_level(f'cluster_{cluster}_family_{family}')
    cf_model.fit(
        train_data=cf_data[cf_data['date'] <= train_end_date],
        val_data=cf_val,
        external_data=external_data
    )
    cf_models[(cluster, family)] = cf_model

hier_forecaster.trained_models['cluster_family'] = cf_models

elapsed = time.time() - start_time
print(f"\nОбучение CatBoost завершено за {elapsed:.1f} секунд")
print(f"Обучено моделей: Total: 1, Clusters: {len(cluster_models)}, Cluster+Family: {len(cf_models)}")


--- Уровень Total ---
    Обучение CatBoost для total
    ✅ Обучение завершено. Использовано итераций: 100

--- Уровень Clusters ---
  Кластер 1 (1/17)
    Обучение CatBoost для cluster_1
    ✅ Обучение завершено. Использовано итераций: 100
  Кластер 2 (2/17)
    Обучение CatBoost для cluster_2
    ✅ Обучение завершено. Использовано итераций: 100
  Кластер 3 (3/17)
    Обучение CatBoost для cluster_3
    ✅ Обучение завершено. Использовано итераций: 100
  Кластер 4 (4/17)
    Обучение CatBoost для cluster_4
    ✅ Обучение завершено. Использовано итераций: 100
  Кластер 5 (5/17)
    Обучение CatBoost для cluster_5
    ✅ Обучение завершено. Использовано итераций: 100
  Кластер 6 (6/17)
    Обучение CatBoost для cluster_6
    ✅ Обучение завершено. Использовано итераций: 100
  Кластер 7 (7/17)
    Обучение CatBoost для cluster_7
    ✅ Обучение завершено. Использовано итераций: 100
  Кластер 8 (8/17)
    Обучение CatBoost для cluster_8
    ✅ Обучение завершено. Использовано итераций: 100
  

KeyboardInterrupt: 

In [ ]:
def predict_catboost_aggregated(forecaster, horizon, start_date, external_data=None):
    """Получает прогнозы от всех обученных CatBoost моделей"""
    
    forecasts = {
        'total': None,
        'clusters': {},
        'cluster_family': {},
        'dates': pd.date_range(start=start_date, periods=horizon)
    }
    
    # Total level
    total_model = forecaster.trained_models['total']
    total_last_data = forecaster.aggregated_series['total'].rename(
        columns={'total_sales': 'value'}
    ).tail(60)
    forecasts['total'] = total_model.predict(horizon, total_last_data, external_data)
    
    # Cluster level
    for cluster, model in forecaster.trained_models['clusters'].items():
        cluster_last_data = forecaster.aggregated_series['by_cluster'][
            forecaster.aggregated_series['by_cluster']['cluster'] == cluster
        ].rename(columns={'unit_sales': 'value'}).tail(60)
        forecasts['clusters'][cluster] = model.predict(horizon, cluster_last_data, external_data)
    
    # Cluster+Family level
    for (cluster, family), model in forecaster.trained_models['cluster_family'].items():
        cf_last_data = forecaster.aggregated_series['by_cluster_family'][
            (forecaster.aggregated_series['by_cluster_family']['cluster'] == cluster) &
            (forecaster.aggregated_series['by_cluster_family']['family'] == family)
        ].rename(columns={'unit_sales': 'value'}).tail(60)
        forecasts['cluster_family'][(cluster, family)] = model.predict(horizon, cf_last_data, external_data)
    
    return forecasts

In [ ]:
start_date = val_data['date'].min()
forecasts = predict_catboost_aggregated(
    hier_forecaster, 
    horizon, 
    start_date,
    external_data=external_data
)

    Прогнозирование для total, признаков: 36
    Прогнозирование для cluster_1, признаков: 40
    Прогнозирование для cluster_2, признаков: 40
    Прогнозирование для cluster_3, признаков: 40
    Прогнозирование для cluster_4, признаков: 40
    Прогнозирование для cluster_5, признаков: 40
    Прогнозирование для cluster_6, признаков: 40
    Прогнозирование для cluster_7, признаков: 40
    Прогнозирование для cluster_8, признаков: 40
    Прогнозирование для cluster_9, признаков: 40
    Прогнозирование для cluster_10, признаков: 40
    Прогнозирование для cluster_11, признаков: 40
    Прогнозирование для cluster_12, признаков: 40
    Прогнозирование для cluster_13, признаков: 40
    Прогнозирование для cluster_14, признаков: 40
    Прогнозирование для cluster_15, признаков: 40
    Прогнозирование для cluster_16, признаков: 40
    Прогнозирование для cluster_17, признаков: 40
    Прогнозирование для cluster_1_family_AUTOMOTIVE, признаков: 41
    Прогнозирование для cluster_1_family_BABY C

In [ ]:
# Дизагрегируем прогнозы
start_time = time.time()

val_pairs = list(val_data.groupby(['store_nbr', 'item_nbr']).groups.keys())
print(f"Всего уникальных пар в валидации: {len(val_pairs)}")

val_predictions = hier_forecaster.disaggregate_to_store_item(
    forecasts=forecasts,
    test_pairs=val_pairs
)

elapsed = time.time() - start_time
print(f"Дизагрегация завершена за {elapsed:.1f} секунд")
print(f"Получено прогнозов: {len(val_predictions)}")

Всего уникальных пар в валидации: 158636
Дизагрегация прогнозов до уровня магазин-товар...
  Средние продажи (total): 8.17
  Форма total прогноза: (16,)
  Прогресс: 0.0% (0/158636)
  Прогресс: 10.0% (15863/158636)
  Прогресс: 20.0% (31726/158636)
  Прогресс: 30.0% (47589/158636)
  Прогресс: 40.0% (63452/158636)
  Прогресс: 50.0% (79315/158636)
  Прогресс: 60.0% (95178/158636)
  Прогресс: 70.0% (111041/158636)
  Прогресс: 80.0% (126904/158636)
  Прогресс: 90.0% (142767/158636)
  Прогресс: 100.0% (158630/158636)
  Дизагрегация завершена. Всего прогнозов: 2538176
Дизагрегация завершена за 284.5 секунд
Получено прогнозов: 2538176


In [ ]:
val_with_pred = val_data.merge(
    val_predictions[['store_nbr', 'item_nbr', 'date', 'predicted']],
    on=['store_nbr', 'item_nbr', 'date'],
    how='left'
)

missing_pred = val_with_pred['predicted'].isna().sum()
if missing_pred > 0:
    print(f"Предупреждение: {missing_pred} записей без прогноза")
    val_with_pred['predicted'] = val_with_pred['predicted'].fillna(0)


Оценка качества...


In [ ]:
# Считаем метрику
cb_score = nwrmsle(
    val_with_pred['unit_sales'].values,
    val_with_pred['predicted'].values,
    val_with_pred['weight'].values
)
print(f"Иерархический CatBoost NWRMSLE: {cb_score:.6f}")


Иерархический CatBoost NWRMSLE: 1.219426


In [ ]:
val_with_pred['cluster'] = val_with_pred['store_nbr'].map(stores.set_index('store_nbr')['cluster'])
val_with_pred['family'] = val_with_pred['item_nbr'].map(items.set_index('item_nbr')['family'])
val_with_pred['perishable'] = val_with_pred['item_nbr'].map(items.set_index('item_nbr')['perishable'])

# По кластерам
print("Ошибка по кластерам магазинов:")
cluster_errors = val_with_pred.groupby('cluster').apply(
    lambda x: nwrmsle(x['unit_sales'].values, x['predicted'].values, x['weight'].values)
).sort_values(ascending=False)
for cluster, error in cluster_errors.head(10).items():
    print(f"  Кластер {cluster}: {error:.6f}")

# По семействам
print("\nОшибка по семействам товаров (топ-5 худших):")
family_errors = val_with_pred.groupby('family').apply(
    lambda x: nwrmsle(x['unit_sales'].values, x['predicted'].values, x['weight'].values)
).sort_values(ascending=False).head(5)
for family, error in family_errors.items():
    print(f"  {family}: {error:.6f}")


АНАЛИЗ ОШИБОК ПО КАТЕГОРИЯМ

Ошибка по кластерам магазинов:
  Кластер 5: 1.479296
  Кластер 6: 1.352843
  Кластер 14: 1.344889
  Кластер 10: 1.279310
  Кластер 11: 1.229944
  Кластер 8: 1.213501
  Кластер 17: 1.202051
  Кластер 3: 1.198949
  Кластер 13: 1.198572
  Кластер 1: 1.177649

Ошибка по семействам товаров (топ-10 худших):
  PRODUCE: 1.612121
  PREPARED FOODS: 1.541623
  POULTRY: 1.472799
  MEATS: 1.433783
  BEVERAGES: 1.356416
  BREAD/BAKERY: 1.313501
  EGGS: 1.258679
  GROCERY I: 1.210204
  DAIRY: 1.142069
  DELI: 1.132885

Ошибка по типу товаров:
  Обычные: 1.163569
  Скоропортящиеся: 1.364321


In [ ]:
# Выберем случайную пару для визуализации
example_store = 3
example_item = 103665
example_data = val_with_pred[
    (val_with_pred['store_nbr'] == example_store) & 
    (val_with_pred['item_nbr'] == example_item)
].sort_values('date')

if len(example_data) > 0:
    plt.figure(figsize=(15, 6))
    plt.plot(example_data['date'], example_data['unit_sales'], 'b-', label='Факт', linewidth=2, marker='o')
    plt.plot(example_data['date'], example_data['predicted'], 'r--', label='Прогноз CatBoost', linewidth=2, marker='s')
    plt.title(f'Магазин {example_store}, Товар {example_item}')
    plt.xlabel('Дата')
    plt.ylabel('Продажи')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()